In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 30 samples and 12123 columns.
First few rows of the data:


,OS.time,OS,Hypoxia_DEG_Score,Gender,Grouping,sample_title,disease state,tissue,treatment,SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,2.504110,1,7.195420,Male,High,sample49,hepatocellular carcinoma,tumor,without treatment,2.130419,...,8.387673,7.346941,8.828142,8.576167,7.499479,7.363618,9.406952,7.648731,6.641186,8.360811
1,0.706849,1,46.831832,Male,High,sample67,hepatocellular carcinoma,tumor,without treatment,1.807826,...,8.131771,6.876079,8.426749,8.604203,7.020778,6.421280,8.775683,7.273592,6.598649,7.762321
2,5.087671,0,47.358300,Male,High,sample68,hepatocellular carcinoma,tumor,without treatment,2.453005,...,8.315868,6.125461,8.025927,8.263996,7.395243,5.763364,8.514467,6.373429,5.374012,7.467705
3,4.980822,0,-26.489002,Male,Low,sample69,hepatocellular carcinoma,tumor,without treatment,2.262247,...,9.497874,8.197639,9.706256,9.848717,8.269399,7.758245,10.048666,8.983378,8.456078,9.127606
4,4.975342,0,-45.116475,Male,Low,sample70,hepatocellular carcinoma,tumor,without treatment,2.169161,...,9.350911,8.134647,9.595574,9.734664,7.856962,7.492528,9.689079,8.695094,7.941265,9.449073


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for OS.time or OS: 0
Number of samples with complete data for both variables: 30
Number of censored samples (event_var == 0.0): 19
Number of events (event_var == 1.0): 11
Censored ratio (among complete): 19 / 30 = 0.633 (63.3%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE201868/description.json
